# 10 MOFs: geometry and CO2 uptake

This notebook selects ten MOFs with complete geometry and CO2 uptake data at all five pressures from the cleaned project dataset, then saves the table as `mof_co2_geometry_10.csv` in this folder. Uptake is reported in mol/kg.

In [2]:
from pathlib import Path

import pandas as pd

# Locate the project root whether this notebook is run from the repo or this folder.
project_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "data/processed/data_clean_v2").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate data/processed/data_clean_v2 from the current directory."
    )

source_path = project_root / "data/processed/data_clean_v2"
output_path = project_root / "notebook/sample/mof_co2_geometry_10.csv"

geometry_columns = ["lcd", "pld", "void_fraction", "surface_area_m2g"]
uptake_columns = [
    "CO2_uptake_0.01bar_molkg",
    "CO2_uptake_0.05bar_molkg",
    "CO2_uptake_0.1bar_molkg",
    "CO2_uptake_0.5bar_molkg",
    "CO2_uptake_2.5bar_molkg",
]
table_columns = ["filename", *geometry_columns, *uptake_columns]

data = pd.read_csv(source_path, low_memory=False)
missing_columns = [column for column in table_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f"Source dataset is missing required columns: {missing_columns}")

mof_table = data[table_columns].dropna().head(10).copy()
if len(mof_table) != 10:
    raise ValueError(f"Expected 10 complete MOFs, but found {len(mof_table)}.")

mof_table.to_csv(output_path, index=False)
print(f"Saved {len(mof_table)} MOFs to {output_path}")
display(mof_table)

Saved 10 MOFs to /home/susan/mof-co2-adsorption/notebook/sample/mof_co2_geometry_10.csv


,filename,lcd,pld,void_fraction,surface_area_m2g,CO2_uptake_0.01bar_molkg,CO2_uptake_0.05bar_molkg,CO2_uptake_0.1bar_molkg,CO2_uptake_0.5bar_molkg,CO2_uptake_2.5bar_molkg
0,hMOF-0,11.75,10.75,0.795539,3676.3,0.022406,0.073552,0.225851,0.885221,4.33307
1,hMOF-1,4.75,3.25,0.388986,580.7,0.173407,0.925813,1.806490,4.779010,7.34618
2,hMOF-10,4.75,3.75,0.307505,377.6,0.651164,2.129830,2.817220,3.909470,4.93236
3,hMOF-100,9.25,8.25,0.731919,3399.2,0.031321,0.143123,0.332034,1.756380,6.91190
4,hMOF-1000,3.75,2.75,0.323803,85.2,0.144024,0.698367,1.402230,2.694200,4.15694
5,hMOF-10000,3.75,2.75,0.194771,17.7,0.239830,0.600202,1.482840,2.374780,2.61176
6,hMOF-10001,6.25,5.25,0.651894,2440.8,0.062156,0.390535,0.552799,4.145720,9.93608
7,hMOF-10003,12.25,10.75,0.818676,3893.8,0.020223,0.116735,0.321263,1.261200,5.07845
8,hMOF-10004,6.25,3.75,0.523720,1089.3,0.198655,1.007990,1.778510,4.220500,6.78177
9,hMOF-10005,8.75,7.75,0.733001,2954.6,0.091681,0.266074,0.511078,2.930230,9.25166


## Copy the matching CIF structure files

The CIF files are stored under `data/raw/hmof_json_local` and use the same MOF IDs as the table. This cell copies the ten matching structures into `notebook/sample/cif_files`.

In [ ]:
import shutil

cif_source_dir = project_root / "data/raw/hmof_json_local"
cif_output_dir = project_root / "notebook/sample/cif_files"
cif_output_dir.mkdir(parents=True, exist_ok=True)

for mof_id in mof_table["filename"]:
    source_cif = cif_source_dir / f"{mof_id}.cif"
    if not source_cif.is_file():
        raise FileNotFoundError(f"CIF file not found for {mof_id}: {source_cif}")
    shutil.copy2(source_cif, cif_output_dir / source_cif.name)

print(f"Copied {len(mof_table)} CIF files to {cif_output_dir}")